In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from matplotlib.ticker import MultipleLocator

In [ ]:
def get_miliseconds(time_str):
    parts = time_str.split(':')
    hours = int(parts[0])
    minutes = int(parts[1])
    seconds_ms = parts[2].split('.')
    seconds = int(seconds_ms[0])
    milliseconds = int(seconds_ms[1])
    time_ms = (hours * 3600 + minutes * 60 + seconds) * 1000 + milliseconds
    return time_ms 



def extract_timings(log_file_path):
    preprocessing_times = []
    preprocessing_ms = []
    odometry_times = []
    odometry_ms =[]
    localization_times = []
    localization_ms = []
    stereo_matches = []
    stereo_matches_ms = []
    keypoints_found = []
    keypoints_found_ms = []
    vo_matches_found = []
    vo_matches_found_ms = []
    inliers = []
    inliers_ms = []
    
    # Regular expressions to match the timing lines
    preprocessing_pattern = r'Finished preprocessing: stereo, which takes (\d+)ms'
    odometry_pattern = r'Finished running odometry: stereo, which takes (\d+)ms'
    localization_pattern = r'Finished running localization: stereo, which takes (\d+)ms'
    stereo_matches_pattern = r'Stereo Matches: (\d+)'
    keypoints_pattern = r'number of keypoints found: (\d+)'
    vo_matches_pattern = r'num matches: (\d+)'
    inliers_pattern = r'RansacModule (\d+)'
    
    left_right_check =0

    with open(log_file_path, 'r') as f:
        for line in f:
            # Check for preprocessing timing
            preprocess_match = re.search(preprocessing_pattern, line)
            if preprocess_match:
                preprocessing_times.append(int(preprocess_match.group(1)))
                preprocessing_ms.append(get_miliseconds(line.split()[0])) 
            
            # Check for odometry timing
            odometry_match = re.search(odometry_pattern, line)
            if odometry_match:
                odometry_times.append(int(odometry_match.group(1)))
                odometry_ms.append(get_miliseconds(line.split()[0])) 
            
            # Check for localization timing
            localization_match = re.search(localization_pattern, line)
            if localization_match:
                localization_times.append(int(localization_match.group(1)))
                localization_ms.append(get_miliseconds(line.split()[0])) 

            # Check num stereo matches
            stereo_match = re.search(stereo_matches_pattern, line)
            if stereo_match:
                stereo_matches.append(int(stereo_match.group(1)))
                stereo_matches_ms.append(get_miliseconds(line.split()[0]))

            # Check num keypoints detected
            keypoints_match = re.search(keypoints_pattern, line)
            if keypoints_match:
                if left_right_check %2 ==0:
                    keypoints_found.append(int(keypoints_match.group(1)))
                    keypoints_found_ms.append(get_miliseconds(line.split()[0]))
                left_right_check +=1
            
            # Check num vertex matches
            vertex_match = re.search(vo_matches_pattern, line)
            if vertex_match:
                vo_matches_found.append(int(vertex_match.group(1)))
                vo_matches_found_ms.append(get_miliseconds(line.split()[0]))

            # RANSAC inliers
            inliers_match = re.search(inliers_pattern, line)
            if inliers_match:
                inliers.append(int(inliers_match.group(1)))
                inliers_ms.append(get_miliseconds(line.split()[0]))
    
    return {
        'preprocessing_times': np.array(preprocessing_times),
        'preprocessing_ms': np.array(preprocessing_ms),
        'odometry_times': np.array(odometry_times),
        'odometry_times_ms': np.array(odometry_ms),
        'localization_times': np.array(localization_times),
        'localization_ms': np.array(localization_ms),
        'stereo_matches': np.array(stereo_matches),
        'stereo_matches_ms': np.array(stereo_matches_ms),
        'keypooints_found': np.array(keypoints_found),
        'keypooints_found_ms': np.array(keypoints_found_ms),
        'vo_matches_found': np.array(vo_matches_found),
        'vo_matches_found_ms': np.array(vo_matches_found_ms),
        'inliers': np.array(inliers),
        'inliers_ms': np.array(inliers_ms)
    }


def print_statistics(timings):
    """Print statistics for each timing category."""
    for category, times in timings.items():
        if times.any():
            print(f"\n{category.replace('_', ' ').title()}:")
            print(f"  Count: {len(times)}")
            print(f"  Min: {min(times)}ms")
            print(f"  Max: {max(times)}ms")
            print(f"  Average: {sum(times)/len(times):.2f}ms")
        else:
            print(f"\n{category.replace('_', ' ').title()}: No data found")


def normalize_times(timings):
    start_time = timings[0]
    norm_times =[]
    for key in timings:
        norm_times.append((key - start_time)/1000.0)
    return norm_times


def preprocess_and_odom(timings):
    new_times =[]
    count =0
    for preproc in timings['preprocessing_times']:
        if count <= len(timings['odometry_times']) -1:
            new_times.append(timings['odometry_times'][count] + preproc)
        else:
            new_times.append(preproc)
        count +=1
    return new_times

In [ ]:
log_file_1 = "/home/adam/Desktop/CurrentBranch/Debug/Hessian_200.log"
log_file_2 = "/home/adam/Desktop/CurrentBranch/Debug/Hessian_400.log"
log_file_3 = "/home/adam/Desktop/CurrentBranch/Debug/Hessian_600.log"
log_file_4 = "/home/adam/Desktop/CurrentBranch/Debug/Hessian_800.log"
log_file_5 = "/home/adam/Desktop/CurrentBranch/Debug/Hessian_1000.log"

timings_200 = extract_timings(log_file_1)
timings_400 = extract_timings(log_file_2)
timings_600 = extract_timings(log_file_3)
timings_800 = extract_timings(log_file_4)
timings_1000 = extract_timings(log_file_5)

# print_statistics(timings_800)

new_timings_200 = preprocess_and_odom(timings_200)
new_timings_400 = preprocess_and_odom(timings_400)
new_timings_600 = preprocess_and_odom(timings_600)
new_timings_800 = preprocess_and_odom(timings_800)
new_timings_1000 = preprocess_and_odom(timings_1000)

# print(new_timings_200)

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
# print(timings['preprocessing_times'].size)

# x1 = np.linspace(1, timings['preprocessing_times'].size, timings['preprocessing_times'].size)

x1 = normalize_times(timings_200['preprocessing_ms'])
plt.plot(x1[1:], timings_200['preprocessing_times'][1:], linewidth=0.5,  label = 'Hesssian = 200')

x2 = normalize_times(timings_400['preprocessing_ms'])
plt.plot(x2[1:], new_timings_400[1:], linewidth=0.5,  label = 'Hesssian = 400')

x3 = normalize_times(timings_600['preprocessing_ms'])
plt.plot(x3[1:], timings_600['preprocessing_times'][1:], linewidth=0.5,  label = 'Hesssian = 600')

# x4 = normalize_times(timings_800['preprocessing_ms'])
# plt.plot(x4[1:], new_timings_800[1:], linewidth=0.5,  label = 'Hesssian = 800')


x5 = normalize_times(timings_1000['preprocessing_ms'])
plt.plot(x5[1:], new_timings_1000[1:], linewidth=0.5,  label = 'Hesssian = 1000')


# x1 = normalize_times(timings_200['preprocessing_ms'])
# plt.plot(x1[1:], timings_200['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 200')

# x2 = normalize_times(timings_400['preprocessing_ms'])
# plt.plot(x2[1:], timings_400['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 400')

# x3 = normalize_times(timings_600['preprocessing_ms'])
# plt.plot(x3[1:], timings_600['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 600')

# x4 = normalize_times(timings_800['preprocessing_ms'])
# plt.plot(x4[1:], timings_800['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 800')

# x5 = normalize_times(timings_1000['preprocessing_ms'])
# plt.plot(x5[1:], timings_1000['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 1000')


plt.xlabel('Time [s]')
plt.ylabel('Computation Time [ms]')
plt.legend()
plt.title('Preprocessing + Mapping time')
plt.show()

### Pre-processing Time

In [ ]:
%matplotlib ipympl
fig, ax = plt.subplots(1, 1, figsize =(16, 5), tight_layout = True)
ax.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 

x1 = normalize_times(timings_200['preprocessing_ms'][25:])
ax.plot(x1[1:], timings_200['preprocessing_times'][1:][25:], linewidth=0.7,  label = "4924") #'Hessian = 200')

x2 = normalize_times(timings_400['preprocessing_ms'][25:])
ax.plot(x2[1:], timings_400['preprocessing_times'][1:][25:], linewidth=0.7,  label = "3210") #'Hessian = 400')

x3 = normalize_times(timings_600['preprocessing_ms'][25:])
ax.plot(x3[1:], timings_600['preprocessing_times'][1:][25:], linewidth=0.7,  label = "2382")#'Hessian = 600')

# x4 = normalize_times(timings_800['preprocessing_ms'])
# plt.plot(x4[1:], timings_800['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 800')

x5 = normalize_times(timings_1000['preprocessing_ms'][25:])
ax.plot(x5[1:], timings_1000['preprocessing_times'][1:][25:], linewidth=0.7,  label = "1602")#'Hessian = 1000')

ax.set_xlim(0, 90)

handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles, labels, ncol=1, frameon=True, fontsize = 22, title_fontsize=14 ,edgecolor='black') #, handlelengt

ax.legend(handles, labels, ncol=1, frameon=True, fontsize=17, title_fontsize=18, edgecolor='black', bbox_to_anchor=(1.01, 1), loc='upper left',  borderaxespad=0, title = "Avg. Features")

ax.xaxis.set_minor_locator(MultipleLocator(2))  # Minor tick every 0.5 units
ax.yaxis.set_minor_locator(MultipleLocator(5))

ax.xaxis.set_major_locator(MultipleLocator(10))    # Major tick every 1 unit
ax.yaxis.set_major_locator(MultipleLocator(20))

ax.set_xlabel('Time [s]', fontsize=25)     # X-axis label size
ax.set_ylabel('Computation Time [ms]', fontsize=25)     # Y-axis label size
ax.tick_params(axis='both', labelsize=20)


# plt.title('Preprocessing time')
plt.show()

In [ ]:
#print_statistics(timings_200)      # 182
# print_statistics(timings_400)    # 159
#print_statistics(timings_600)     # 125
# print_statistics(timings_1000)   # 81

### Odometry Time

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
# print(timings['odometry_times'].size)
# x1 = np.linspace(1, timings['odometry_times'].size, timings['odometry_times'].size)
# x1 = normalize_times(timings_200['odometry_times_ms'])
# plt.plot(x1, timings_200['odometry_times'], linewidth=0.5,  label = 'Hessian = 200')

x2 = normalize_times(timings_400['odometry_times_ms'])
plt.plot(x2, timings_400['odometry_times'], linewidth=0.5,  label = 'Hessian = 400')

x3 = normalize_times(timings_600['odometry_times_ms'])
plt.plot(x3, timings_600['odometry_times'], linewidth=0.5,  label = 'Hessian = 600')

x4 = normalize_times(timings_800['odometry_times_ms'])
plt.plot(x4, timings_800['odometry_times'], linewidth=0.5,  label = 'Hessian = 800')

x5 = normalize_times(timings_1000['odometry_times_ms'])
plt.plot(x5, timings_1000['odometry_times'], linewidth=0.5,  label = 'Hessian = 1000')

plt.xlabel('Frame')
plt.ylabel('Time [ms]')
plt.legend()
plt.title('Odometry time')
plt.show()

### Feature Numbers over pipeline

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(12, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(timings['stereo_matches'].size)
x1 = np.linspace(1, timings['stereo_matches'].size, timings['stereo_matches'].size)
plt.plot(x1, timings['stereo_matches'], linewidth=0.5,  label = 'Stereo Matches')

# Add your second dataset here
x2 = np.linspace(1, timings['keypooints_found'].size, timings['keypooints_found'].size)
plt.plot(x2, timings['keypooints_found'], linewidth=0.5, label='keypooints Detected')

# Add your second dataset here
x3 = np.linspace(1, timings['vo_matches_found'].size, timings['vo_matches_found'].size)
plt.plot(x3, timings['vo_matches_found'], linewidth=0.5, label='Vertex Matches')

# Add your second dataset here
x4 = np.linspace(1, timings['inliers'].size, timings['inliers'].size)
plt.plot(x4, timings['inliers'], linewidth=0.5, label='RANSAC Inliers')


plt.xlabel('Frame')
plt.ylabel('num Features')
plt.legend()
plt.title('Keypoints Detected and Steero Matches')
plt.show()

### Inliers

In [ ]:
%matplotlib ipympl
fig, ax = plt.subplots(1, 1, figsize =(16, 5), tight_layout = True)
ax.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 

x1 = normalize_times(timings_200['inliers_ms'][25:])
ax.plot(x1, timings_200['inliers'][25:], linewidth=0.7,  label = "4924") #'Hessian = 200')

x2 = normalize_times(timings_400['inliers_ms'][25:])
ax.plot(x2, timings_400['inliers'][25:], linewidth=0.7,  label = "3210") #'Hessian = 400')

x3 = normalize_times(timings_600['inliers_ms'][25:])
ax.plot(x3, timings_600['inliers'][25:], linewidth=0.7,  label = "2382")#'Hessian = 600')

# x4 = normalize_times(timings_800['preprocessing_ms'])
# plt.plot(x4[1:], timings_800['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 800')

x5 = normalize_times(timings_1000['inliers_ms'][25:])
ax.plot(x5, timings_1000['inliers'][25:], linewidth=0.7,  label = "1602")#'Hessian = 1000')


handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles, labels, ncol=1, frameon=True, fontsize = 22, title_fontsize=14 ,edgecolor='black') #, handlelengt

ax.legend(handles, labels, ncol=1, frameon=True, fontsize=17, title_fontsize=18, edgecolor='black', bbox_to_anchor=(1.01, 1), loc='upper left',  borderaxespad=0, title = "Avg. Features")

ax.set_xlim(0, 90)

ax.xaxis.set_minor_locator(MultipleLocator(2))  # Minor tick every 0.5 units
ax.yaxis.set_minor_locator(MultipleLocator(10))

ax.xaxis.set_major_locator(MultipleLocator(10))    # Major tick every 1 unit
ax.yaxis.set_major_locator(MultipleLocator(100))

ax.set_xlabel('Time [s]', fontsize=25)     # X-axis label size
ax.set_ylabel('Number of Inliers', fontsize=25)     # Y-axis label size
ax.tick_params(axis='both', labelsize=20)

plt.show()

In [ ]:
%matplotlib ipympl
fig, ax = plt.subplots(1, 1, figsize =(16, 5), tight_layout = True)
ax.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 

x1 = normalize_times(timings_200['preprocessing_ms'])
ax.plot(x1[1:], timings_200['preprocessing_times'][1:], linewidth=0.7,  label = "4924") #'Hessian = 200')

x2 = normalize_times(timings_400['preprocessing_ms'])
ax.plot(x2[1:], timings_400['preprocessing_times'][1:], linewidth=0.7,  label = "3210") #'Hessian = 400')

x3 = normalize_times(timings_600['preprocessing_ms'])
ax.plot(x3[1:], timings_600['preprocessing_times'][1:], linewidth=0.7,  label = "2382")#'Hessian = 600')

# x4 = normalize_times(timings_800['preprocessing_ms'])
# plt.plot(x4[1:], timings_800['preprocessing_times'][1:], linewidth=0.5,  label = 'Hessian = 800')

x5 = normalize_times(timings_1000['preprocessing_ms'])
ax.plot(x5[1:], timings_1000['preprocessing_times'][1:], linewidth=0.7,  label = "1602")#'Hessian = 1000')



handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles, labels, ncol=1, frameon=True, fontsize = 22, title_fontsize=14 ,edgecolor='black') #, handlelengt

ax.legend(handles, labels, ncol=1, frameon=True, fontsize=17, title_fontsize=18, edgecolor='black', bbox_to_anchor=(1.01, 1), loc='upper left',  borderaxespad=0, title = "Avg. Features")

ax.xaxis.set_minor_locator(MultipleLocator(2))  # Minor tick every 0.5 units
ax.yaxis.set_minor_locator(MultipleLocator(5))

ax.xaxis.set_major_locator(MultipleLocator(10))    # Major tick every 1 unit
ax.yaxis.set_major_locator(MultipleLocator(20))

ax.set_xlabel('Time [s]', fontsize=25)     # X-axis label size
ax.set_ylabel('Computation Time [ms]', fontsize=25)     # Y-axis label size
ax.tick_params(axis='both', labelsize=20)


# plt.title('Preprocessing time')
plt.show()

In [ ]:
%matplotlib ipympl
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
print(timings['localization_times'].size)
x1 = np.linspace(1, timings['localization_times'].size, timings['localization_times'].size)
plt.plot(x1, timings['localization_times'], linewidth=0.5,  label = 'Repeat 1')

plt.xlabel('Frame')
plt.ylabel('Time [ms]')
plt.legend()
plt.title('localization time')
plt.show()